<a href="https://colab.research.google.com/github/Goat-Seller/bankruptcy-prediction/blob/main/regression_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Taiwanese companies bankrupcy prediction

#1 - data preparation

In [21]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme()

data = pd.read_csv('/content/data.csv')

## Data split

In [22]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

target = 'Bankrupt?'

x = data.drop(columns=[target])
y = data[target]

x_train, x_rest, y_train, y_rest = train_test_split(x, y, test_size=0.6, random_state=1, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_rest, y_rest, test_size=0.5  , random_state=1, stratify=y_rest)

In [23]:
required_features = [
    'ROA(A) before interest and % after tax', #
    'Research and development expense rate', #
    'Total debt/Total net worth', #
    'Debt ratio %', #
    'Cash Flow to Liability', #
    'Revenue Per Share (Yuan ¥)', #
    'Current Liabilities/Equity', #
    'Net Income to Total Assets', #
    'Borrowing dependency', #
    'Allocation rate per person', #
]

chosen_features = [
    'Persistent EPS in the Last Four Seasons',
    'Total debt/Total net worth',
    'Non-industry income and expenditure/revenue',
    'Net Value Growth Rate',
    'Quick Ratio',
    'Tax rate (A)',
    'Cash/Total Assets',
    'Operating Gross Margin',
]


# Logistic Regression

In [36]:
x_train.columns = x_train.columns.str.strip()
x_val.columns = x_val.columns.str.strip()

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score

scaler = StandardScaler()

x_train_s_chosen = scaler.fit_transform(x_train[chosen_features])
x_val_s_chosen = scaler.transform(x_val[chosen_features])
x_train_s_required = scaler.fit_transform(x_train[required_features])
x_val_s_required = scaler.transform(x_val[required_features])
x_train_s_all = scaler.fit_transform(x_train)
x_val_s_all = scaler.transform(x_val)

feature_sets = [
    ("Chosen", x_train_s_chosen, x_val_s_chosen),
    ("Required", x_train_s_required, x_val_s_required),
    ("All", x_train_s_all, x_val_s_all),
]


for set_name, x_tr, x_va in feature_sets:
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(x_tr, y_train)

    y_pred_train = clf.predict(x_tr)
    y_pred_val = clf.predict(x_va)

    print(set_name, "features:")
    print("Accuracy train: ", accuracy_score(y_train, y_pred_train))
    print("Accuracy val: ", accuracy_score(y_val, y_pred_val))
    print("Recall val: ", recall_score(y_val, y_pred_val))
    print("\n")

Chosen features:
Accuracy train:  0.8104143747708105
Accuracy val:  0.8054740957966764
Recall val:  0.8939393939393939


Required features:
Accuracy train:  0.8533186651998533
Accuracy val:  0.8455522971652004
Recall val:  0.8939393939393939


All features:
Accuracy train:  0.8841217455078841
Accuracy val:  0.8626588465298143
Recall val:  0.7575757575757576




## SVM

In [45]:
from sklearn import svm

x_train_s_chosen = scaler.fit_transform(x_train[chosen_features])
x_val_s_chosen = scaler.transform(x_val[chosen_features])
x_train_s_required = scaler.fit_transform(x_train[required_features])
x_val_s_required = scaler.transform(x_val[required_features])
x_train_s_all = scaler.fit_transform(x_train)
x_val_s_all = scaler.transform(x_val)

feature_sets = [
    ("Chosen", x_train_s_chosen, x_val_s_chosen),
    ("Required", x_train_s_required, x_val_s_required),
    ("All", x_train_s_all, x_val_s_all),
]

for set_name, x_tr, x_va in feature_sets:
    for kernel in ["linear", "poly", "rbf", "sigmoid"]:
        clf = svm.SVC(kernel=kernel, class_weight="balanced")
        clf.fit(x_tr, y_train)
        pred_tr = clf.predict(x_tr)
        pred_va = clf.predict(x_va)
        print(f"{set_name:9s} {kernel:8s} train {accuracy_score(y_train, pred_tr):.3f} | "
              f"val {accuracy_score(y_val, pred_va):.3f} | recall {recall_score(y_val, pred_va):.3f}")

Chosen    linear   train 0.806 | val 0.805 | recall 0.894
Chosen    poly     train 0.879 | val 0.862 | recall 0.773
Chosen    rbf      train 0.825 | val 0.810 | recall 0.803
Chosen    sigmoid  train 0.736 | val 0.732 | recall 0.864
Required  linear   train 0.862 | val 0.848 | recall 0.848
Required  poly     train 0.931 | val 0.925 | recall 0.667
Required  rbf      train 0.863 | val 0.853 | recall 0.773
Required  sigmoid  train 0.736 | val 0.731 | recall 0.742
All       linear   train 0.886 | val 0.859 | recall 0.773
All       poly     train 0.957 | val 0.929 | recall 0.576
All       rbf      train 0.938 | val 0.911 | recall 0.606
All       sigmoid  train 0.797 | val 0.796 | recall 0.788
